In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [ ]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(model=model)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.10323123633861542
epoch:  1, loss: 0.07066012173891068
epoch:  2, loss: 0.05337714031338692
epoch:  3, loss: 0.04422901198267937
epoch:  4, loss: 0.039407290518283844
epoch:  5, loss: 0.03687683865427971
epoch:  6, loss: 0.035553716123104095
epoch:  7, loss: 0.03486373648047447
epoch:  8, loss: 0.0345044806599617
epoch:  9, loss: 0.03431738540530205
epoch:  10, loss: 0.03421969339251518
epoch:  11, loss: 0.03416832908987999
epoch:  12, loss: 0.034140992909669876
epoch:  13, loss: 0.03412606194615364
epoch:  14, loss: 0.03411757946014404
epoch:  15, loss: 0.034112416207790375
epoch:  16, loss: 0.03410559892654419
epoch:  17, loss: 0.034098222851753235
epoch:  18, loss: 0.03409377858042717
epoch:  19, loss: 0.034088265150785446
epoch:  20, loss: 0.03408178314566612
epoch:  21, loss: 0.03407786414027214
epoch:  22, loss: 0.03407294303178787
epoch:  23, loss: 0.034067168831825256
epoch:  24, loss: 0.03406365215778351
epoch:  25, loss: 0.0340583436191082
epoch:  26, loss:

In [6]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.8129745125770569
Test metrics:  R2 = 0.7260271310806274
